# Пробный прогон: детекция крупнее и другие модели внешности

Считаем только окна с толпой и сложными случаями, всю запись повторим, только если станет лучше.

1. Меню **Среда выполнения → Сменить среду → GPU**. Хватит T4 или L4, A100 быстрее, но быстрее тратит вычислительные единицы.
2. На Google Drive создайте папку `mct` (если её ещё нет) и положите в неё `mct_colab.zip`. Файл лежит в проекте: `artifacts/mct_colab.zip`.
3. Меню **Среда выполнения → Выполнить все**, разрешите доступ к Диску.
4. Если сессия оборвётся, запустите всё снова: готовое будет пропущено.
5. Когда появится надпись **ГОТОВО**, скачайте с Диска `mct/trial_results.zip` и положите в папку проекта. Архив обновляется после каждого этапа, поэтому готовое можно забрать и раньше.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi --query-gpu=name,memory.total --format=csv
!ls -la /content/drive/MyDrive/mct/mct_colab.zip


In [ ]:
%%bash
set -e
mkdir -p /content/mct
unzip -oq /content/drive/MyDrive/mct/mct_colab.zip -d /content/mct
pip install -q boxmot==26.0.0 ultralytics==8.4.203 imageio-ffmpeg lap pycocotools pyarrow
ls /content/mct /content/mct/data


In [ ]:
import torch, boxmot, ultralytics
print('cuda', torch.cuda.is_available(), 'boxmot', boxmot.__version__, 'ultralytics', ultralytics.__version__)
assert torch.cuda.is_available(), 'GPU не подключён: смените среду выполнения'


In [ ]:
import os, shutil, subprocess
from pathlib import Path

OUT = Path('/content/drive/MyDrive/mct/trial')
OUT.mkdir(parents=True, exist_ok=True)
ENV = {**os.environ, 'PYTHONPATH': 'src'}

def run(*args):
    done = subprocess.run(['python', *map(str, args)], cwd='/content/mct', env=ENV)
    if done.returncode:
        print('ОШИБКА, код', done.returncode, args, flush=True)

def pack():
    # После каждого этапа: готовое можно забрать, не дожидаясь конца
    archive = shutil.make_archive('/content/drive/MyDrive/mct/trial_results', 'zip', OUT)
    print('Архив:', archive, round(os.path.getsize(archive) / 1e6), 'МБ', flush=True)


## Другие модели внешности

Те же снимки, что в готовой детекции: по одному в секунду, с фоном и с фоном, закрашенным по маске. Веса скачает BoxMOT. Около 15 минут.


In [ ]:
run('scripts/reembed.py', 'data', '--out', OUT / 'reid', '--device', '0', '--kinds', 'box', 'mask',
    '--models', 'lmbn_n_market', 'lmbn_n_duke', 'lmbn_n_cuhk03_d', 'osnet_ibn_x1_0_msmt17', 'resnet50_fc512_msmt17')
pack()


## Детекция на окнах записи

Окна по общим часам, секунды от 17:00:00. `crowd`: 17:45:00–17:46:10, толпа, для неё есть ручная разметка. `door`: 17:51:35–17:52:10, сложные случаи у двери. Сейчас в работе `yolo11m-seg`, вход 960. На T4 примерно полчаса на вариант.


In [ ]:
WINDOWS = {'crowd': (2700, 70), 'door': (3095, 35)}
# модель, размер входа сети, во сколько раз крупнее рабочего кадра читать запись
TRIALS = [('yolo11m-seg.pt', 1920, 2), ('yolo11m-seg.pt', 1280, 1), ('yolo11x-seg.pt', 1280, 1)]

for model, imgsz, scale in TRIALS:
    for window, (start, duration) in WINDOWS.items():
        print('===', model, imgsz, window, flush=True)
        run('scripts/run_perception.py', 'data', '--out', OUT / f'{Path(model).stem}_{imgsz}' / window, '--device', '0', '--model', model,
            '--imgsz', imgsz, '--detect-scale', scale, '--clock-start', start, '--duration', duration)
    pack()


In [ ]:
ready = sorted(str(path.relative_to(OUT)) for path in OUT.rglob('*_summary.json'))
print(len(ready), 'камер посчитано из', 5 * len(WINDOWS) * len(TRIALS))
print('ГОТОВО. Скачайте с Диска mct/trial_results.zip и положите в папку проекта.')
